In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

import matplotlib.pyplot as plt

from aiconfigurator.sdk import common
from aiconfigurator.sdk.operations import warm_all_op_data
from aiconfigurator.sdk.perf_database import get_database, PerfDataNotAvailableError

from aiconfigurator.sdk.common import DatabaseMode

# Allow override via env for parameterized testing (e.g. test_validate_database per combo)
system = os.environ.get("AIC_VALIDATE_SYSTEM", "h200_sxm")
backend = os.environ.get("AIC_VALIDATE_BACKEND", "trtllm")
version = os.environ.get("AIC_VALIDATE_VERSION", "current")

database = get_database(system=system, backend=backend, version=version)
if database is None:
    raise RuntimeError(f"No database for system={system!r} backend={backend!r} version={version!r}")

# This notebook walks every op's instance attribute directly (database._gemm_data,
# database._context_attention_data, etc.) for visualization. Op data is loaded
# lazily on first query, so trigger every op up front to restore the
# "everything loaded" semantics this diagnostic tool depends on.
warm_all_op_data(database)

# Per-op reference values (SILICON latencies and SOL_FULL triples) come from
# the compiled Rust engine's ad-hoc op-list evaluation instead of the
# PerfDatabase.query_* per-call stack (scheduled for removal, #1357).
try:
    from tools.sanity_check.engine_reference import EngineReference
except ModuleNotFoundError:
    from engine_reference import EngineReference

reference = EngineReference(database)


In [ ]:
def visualize_gemm(database, reference):
    # gemm
    n_k = [[4096, 4096], [8192, 8192], [8192, 1024], [1024, 8192]]
    m_list = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_k), figsize=(5 * len(n_k), 5 * 2), squeeze=False)
    for i, (n, k) in enumerate(n_k):
        for color_id, quant_mode in enumerate(database._gemm_data.keys()):
            plotted_m_list = []
            sol_math_list = []
            sol_mem_list = []
            for m in m_list:
                try:
                    db_time = reference.query_gemm(
                        m=m, n=n, k=k, quant_mode=quant_mode, database_mode=DatabaseMode.SILICON
                    )
                    sol_time, sol_math, sol_mem = reference.query_gemm(
                        m=m, n=n, k=k, quant_mode=quant_mode, database_mode=DatabaseMode.SOL_FULL
                    )
                except (PerfDataNotAvailableError, ValueError) as e:
                    print(f"Skipping GEMM point m={m}, n={n}, k={k}, quant_mode={quant_mode}: {e}")
                    continue
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                plotted_m_list.append(m)
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            if not plotted_m_list:
                continue
            ax[0, i].plot(plotted_m_list, sol_math_list, color=color_list[color_id], label=f"{quant_mode} math")
            ax[1, i].plot(
                plotted_m_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{quant_mode} mem",
            )
        ax[0, i].set_title(f"n={n}, k={k}")
        ax[0, i].set_xlabel("num_tokens")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("num_tokens")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_gemm(database, reference)

In [ ]:
def visualize_context_attention(database, reference):
    b = 1
    n = 32
    n_kv_list = [1, 2, 4, 8, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 524288]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2), squeeze=False)
    color_id = 0
    # Non-baseline (fmha, kv) combos may legitimately cover only part of the
    # structural grid: the FP8-prefill relabel records compute-dtype truth, so
    # a system whose standard-GQA fp8-KV prefill computes in BF16 (e.g. SM89
    # flashinfer) ships fp8-COMPUTE rows only for the shapes serving actually
    # dispatches to an fp8-capable kernel (sinks/triton profiles). Probe-and-
    # skip those combos like visualize_gemm does — but keep the gate strong:
    # the (bfloat16, bfloat16) baseline grid must answer every probe.
    skipped_probes = {}
    for i, n_kv in enumerate(n_kv_list):
        for quant_mode in database._context_attention_data.keys():
            for kvcache_quant_mode in database._context_attention_data[quant_mode].keys():
                sol_math_list = []
                sol_mem_list = []
                probed_s_list = []
                for s in s_list:
                    sol_time, sol_math, sol_mem = reference.query_context_attention(
                        b=b,
                        s=s,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                        prefix=0,
                    )
                    try:
                        db_time = reference.query_context_attention(
                            b=b,
                            s=s,
                            n=n,
                            n_kv=n_kv,
                            kvcache_quant_mode=kvcache_quant_mode,
                            fmha_quant_mode=quant_mode,
                            database_mode=DatabaseMode.SILICON,
                            prefix=0,
                        )
                    except PerfDataNotAvailableError as e:
                        combo = (quant_mode, kvcache_quant_mode)
                        skipped_probes.setdefault(combo, []).append((n_kv, s))
                        print(f"Skipping ctx-attn point (fmha={quant_mode}, kv={kvcache_quant_mode}, n_kv={n_kv}, s={s}): {e}")
                        continue
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                    probed_s_list.append(s)
                if not probed_s_list:
                    continue
                ax[0, i].plot(
                    probed_s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    probed_s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.show()

    # Gate: the baseline (bfloat16, bfloat16) grid must have answered every
    # probe — partial coverage there is a real data regression, not a
    # documented compute-dtype gap.
    baseline = (common.FMHAQuantMode.bfloat16, common.KVCacheQuantMode.bfloat16)
    if baseline in skipped_probes:
        raise AssertionError(
            f"(bf16, bf16) context-attention grid failed probes at: {skipped_probes[baseline]}"
        )
    if skipped_probes:
        print("Documented partial fp8-compute grids (probe-and-skip):")
        for combo, points in skipped_probes.items():
            print(f"  {combo}: {len(points)} skipped probe points")


visualize_context_attention(database, reference)

In [ ]:
def visualize_context_attention_with_prefix(database, reference):
    b = 1
    n = 32
    n_kv_list = [1, 2, 4, 8, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 524288]
    prefix_factor = 0.3

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2), squeeze=False)
    color_id = 0
    for i, n_kv in enumerate(n_kv_list):
        for quant_mode in database._context_attention_data.keys():
            for kvcache_quant_mode in database._context_attention_data[quant_mode].keys():
                sol_math_list = []
                sol_mem_list = []
                probed_s_list = []
                for s in s_list:
                    prefix_len = int(s * prefix_factor)
                    sol_time, sol_math, sol_mem = reference.query_context_attention(
                        b=b,
                        s=s - prefix_len,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                        prefix=prefix_len,
                    )
                    # Probe-and-skip like visualize_context_attention: partial
                    # fp8-compute grids may not answer every prefix probe.
                    try:
                        db_time = reference.query_context_attention(
                            b=b,
                            s=s - prefix_len,
                            n=n,
                            n_kv=n_kv,
                            kvcache_quant_mode=kvcache_quant_mode,
                            fmha_quant_mode=quant_mode,
                            database_mode=DatabaseMode.SILICON,
                            prefix=prefix_len,
                        )
                    except PerfDataNotAvailableError as e:
                        print(
                            f"Skipping ctx-attn prefix point (fmha={quant_mode}, kv={kvcache_quant_mode}, "
                            f"n_kv={n_kv}, s={s}): {e}"
                        )
                        continue
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                    probed_s_list.append(s)
                if not probed_s_list:
                    continue
                ax[0, i].plot(
                    probed_s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    probed_s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_context_attention_with_prefix(database, reference)

In [ ]:
def visualize_generation_attention(database, reference):
    b = 64
    n = 32
    n_kv_list = [1, 2, 4, 8]  # mha uses sol in current version
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2), squeeze=False)
    for i, n_kv in enumerate(n_kv_list):
        for color_id, kvcache_quant_mode in enumerate(database._generation_attention_data.keys()):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = reference.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = reference.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(
                s_list,
                sol_math_list,
                color=color_list[color_id],
                label=f"{kvcache_quant_mode} math",
            )
            ax[1, i].plot(
                s_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{kvcache_quant_mode} mem",
            )
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_attention(database, reference)
# visualize_generation_attention(database_fixed, reference)

In [ ]:
def visualize_generation_attention_b(database, reference):
    if not database._generation_attention_data:
        print("No generation attention data available – skipping")
        return

    b_list = [1, 4, 16, 64, 256, 1024]
    n = 8
    n_kv = 2
    s_list = [128, 512, 1024, 4096, 8192, 16384, 32768, 65535]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(
        2,
        len(database._generation_attention_data.keys()),
        figsize=(5 * len(database._generation_attention_data.keys()), 5 * 2),
        squeeze=False,
    )
    for i, kvcache_quant_mode in enumerate(database._generation_attention_data.keys()):
        for color_id, b in enumerate(b_list):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = reference.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = reference.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(s_list, sol_math_list, color=color_list[color_id], label=f"b{b} math")
            ax[1, i].plot(s_list, sol_mem_list, color=color_list[color_id], linestyle="--", label=f"b{b} mem")
        ax[0, i].set_title(f"kvcache={kvcache_quant_mode} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_attention_b(database, reference)

In [ ]:
def visualize_context_mla_with_prefix(database, reference):
    database._context_mla_data.raise_if_not_loaded()

    b = 8
    n_list = [2, 4, 8, 16, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
    prefix_scale = 0.3

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_list), figsize=(5 * len(n_list), 5 * 2), squeeze=False)
    color_id = 0
    for i, n_q in enumerate(n_list):
        for quant_mode in database._context_mla_data.keys():
            for kvcache_quant_mode in database._context_mla_data[quant_mode].keys():
                plotted_s_list = []
                sol_math_list = []
                sol_mem_list = []
                for s in s_list:
                    prefix_len = int(s * prefix_scale)
                    try:
                        sol_time, sol_math, sol_mem = reference.query_context_mla(
                            b=b,
                            s=s - prefix_len,
                            num_heads=n_q,
                            kvcache_quant_mode=kvcache_quant_mode,
                            fmha_quant_mode=quant_mode,
                            database_mode=DatabaseMode.SOL_FULL,
                            prefix=prefix_len,
                        )
                        db_time = reference.query_context_mla(
                            b=b,
                            s=s - prefix_len,
                            num_heads=n_q,
                            kvcache_quant_mode=kvcache_quant_mode,
                            fmha_quant_mode=quant_mode,
                            database_mode=DatabaseMode.SILICON,
                            prefix=prefix_len,
                        )
                    except (PerfDataNotAvailableError, ValueError) as e:
                        print(
                            f"Skipping context MLA point b={b}, s={s}, prefix={prefix_len}, num_heads={n_q}, "
                            f"fmha_quant_mode={quant_mode}, kvcache_quant_mode={kvcache_quant_mode}: {e}"
                        )
                        continue
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    plotted_s_list.append(s)
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                if not plotted_s_list:
                    continue
                ax[0, i].plot(
                    plotted_s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    plotted_s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n_q={n_q}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


try:
    visualize_context_mla_with_prefix(database, reference)
except (PerfDataNotAvailableError, ValueError) as e:
    print(e)

In [ ]:
def visualize_generation_mla(database, reference):
    database._generation_mla_data.raise_if_not_loaded()

    b = 64
    n_list = [2, 4, 8, 16, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_list), figsize=(5 * len(n_list), 5 * 2), squeeze=False)
    for i, n_q_per_gpu in enumerate(n_list):
        for color_id, kvcache_quant_mode in enumerate(database._generation_mla_data.keys()):
            plotted_s_list = []
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                try:
                    sol_time, sol_math, sol_mem = reference.query_generation_mla(
                        b=b,
                        s=s,
                        num_heads=n_q_per_gpu,
                        kvcache_quant_mode=kvcache_quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                    )
                    db_time = reference.query_generation_mla(
                        b=b,
                        s=s,
                        num_heads=n_q_per_gpu,
                        kvcache_quant_mode=kvcache_quant_mode,
                        database_mode=DatabaseMode.SILICON,
                    )
                except (PerfDataNotAvailableError, ValueError) as e:
                    print(
                        f"Skipping generation MLA point b={b}, s={s}, num_heads={n_q_per_gpu}, "
                        f"kvcache_quant_mode={kvcache_quant_mode}: {e}"
                    )
                    continue
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                plotted_s_list.append(s)
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            if not plotted_s_list:
                continue
            ax[0, i].plot(
                plotted_s_list,
                sol_math_list,
                color=color_list[color_id],
                label=f"{kvcache_quant_mode} math",
            )
            ax[1, i].plot(
                plotted_s_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{kvcache_quant_mode} mem",
            )
        ax[0, i].set_title(f"b={b} n_q_per_gpu={n_q_per_gpu}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


try:
    visualize_generation_mla(database, reference)
except (PerfDataNotAvailableError, ValueError) as e:
    print(e)

In [ ]:
def visualize_generation_mla_b(database, reference):
    database._generation_mla_data.raise_if_not_loaded()

    b_list = [1, 4, 16, 64, 256, 1024]
    num_q = 16
    s_list = [128, 512, 1024, 4096, 8192, 16384, 32768]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(
        2,
        len(database._generation_mla_data.keys()),
        figsize=(5 * len(database._generation_mla_data.keys()), 5 * 2),
        squeeze=False,
    )
    for i, kvcache_quant_mode in enumerate(database._generation_mla_data.keys()):
        for color_id, b in enumerate(b_list):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = reference.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=num_q,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = reference.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=num_q,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(s_list, sol_math_list, color=color_list[color_id], label=f"b{b} math")
            ax[1, i].plot(s_list, sol_mem_list, color=color_list[color_id], linestyle="--", label=f"b{b} mem")
        ax[0, i].set_title(f"kvcache={kvcache_quant_mode} n_q_per_gpu={num_q}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


try:
    visualize_generation_mla_b(database, reference)
except (PerfDataNotAvailableError, ValueError) as e:
    print(e)

In [ ]:
def visualize_dsa_module(database, reference):
    b = 1
    num_heads = 128
    # Index dims (index_n_heads=64, index_head_dim=128, index_topk=2048) are
    # derived from the architecture by the op layer.
    architecture = "DeepseekV32ForCausalLM"
    kv_cache_dtype = common.KVCacheQuantMode.bfloat16
    fmha_quant_mode = common.FMHAQuantMode.bfloat16

    context_s_list = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 65536]
    generation_s_list = [128, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536 * 4]

    fig, ax = plt.subplots(2, 2, figsize=(14, 8), squeeze=False)

    context_sol_math_list = []
    context_sol_mem_list = []
    for s in context_s_list:
        sol_time, sol_math, sol_mem = reference.query_context_dsa_module(
            b=b,
            s=s,
            prefix=0,
            num_heads=num_heads,
            architecture=architecture,
            kvcache_quant_mode=kv_cache_dtype,
            fmha_quant_mode=fmha_quant_mode,
            database_mode=DatabaseMode.SOL_FULL,
        )
        db_time = reference.query_context_dsa_module(
            b=b,
            s=s,
            prefix=0,
            num_heads=num_heads,
            architecture=architecture,
            kvcache_quant_mode=kv_cache_dtype,
            fmha_quant_mode=fmha_quant_mode,
            database_mode=DatabaseMode.SILICON,
        )
        context_sol_math_list.append(sol_math / db_time)
        context_sol_mem_list.append(sol_mem / db_time)

    ax[0, 0].plot(context_s_list, context_sol_math_list, color="blue", label="context math")
    ax[0, 1].plot(context_s_list, context_sol_mem_list, color="orange", linestyle="--", label="context mem")
    ax[0, 0].set_title("DSA Context Module: math sol %")
    ax[0, 1].set_title("DSA Context Module: mem sol %")
    ax[0, 0].set_xlabel("s")
    ax[0, 1].set_xlabel("s")
    ax[0, 0].set_ylabel("ratio")
    ax[0, 1].set_ylabel("ratio")
    ax[0, 0].set_ylim(0, 1)
    ax[0, 1].set_ylim(0, 1)
    ax[0, 0].legend()
    ax[0, 1].legend()

    generation_sol_math_list = []
    generation_sol_mem_list = []
    for s in generation_s_list:
        sol_time, sol_math, sol_mem = reference.query_generation_dsa_module(
            b=b,
            s=s,
            num_heads=num_heads,
            architecture=architecture,
            kv_cache_dtype=kv_cache_dtype,
            database_mode=DatabaseMode.SOL_FULL,
        )
        db_time = reference.query_generation_dsa_module(
            b=b,
            s=s,
            num_heads=num_heads,
            architecture=architecture,
            kv_cache_dtype=kv_cache_dtype,
            database_mode=DatabaseMode.SILICON,
        )
        generation_sol_math_list.append(sol_math / db_time)
        generation_sol_mem_list.append(sol_mem / db_time)

    ax[1, 0].plot(generation_s_list, generation_sol_math_list, color="green", label="generation math")
    ax[1, 1].plot(
        generation_s_list,
        generation_sol_mem_list,
        color="red",
        linestyle="--",
        label="generation mem",
    )
    ax[1, 0].set_title("DSA Generation Module: math sol %")
    ax[1, 1].set_title("DSA Generation Module: mem sol %")
    ax[1, 0].set_xlabel("kv cache len (s)")
    ax[1, 1].set_xlabel("kv cache len (s)")
    ax[1, 0].set_ylabel("ratio")
    ax[1, 1].set_ylabel("ratio")
    ax[1, 0].set_ylim(0, 1)
    ax[1, 1].set_ylim(0, 1)
    ax[1, 0].legend()
    ax[1, 1].legend()

    plt.tight_layout()
    plt.show()


# visualize_dsa_module(database, reference)

In [ ]:
try:
    from tools.sanity_check.moe_chart_profiles import MoeChartProfile, select_moe_chart_profiles
except ModuleNotFoundError:
    from moe_chart_profiles import MoeChartProfile, select_moe_chart_profiles


def visualize_moe(database, reference):
    database._moe_data.raise_if_not_loaded()

    workload_distributions = list(list(database._moe_data.values())[0].keys())
    tp_list = [1, 2, 4, 8]
    ep_list = [1, 2, 4, 8, 16, 32]
    max_node_gpus = int(database.system_spec.get("node", {}).get("num_gpus_per_node", 0) or 0)
    tp_ep_list = []
    for tp in tp_list:
        for ep in ep_list:
            if max_node_gpus and tp * ep > max_node_gpus:
                continue
            if database.backend == "vllm" and tp > 1 and ep > 1:
                continue
            if database.backend == "sglang" and ep > 1:
                continue
            if tp * ep >= 4 and tp * ep <= 32:
                tp_ep_list.append([tp, ep])
    if not tp_ep_list:
        print("No supported multi-GPU MoE chart points available - skipping")
        return
    m_list = [1, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
    color_list = list(plt.get_cmap("tab20").colors)
    color_by_series = {}
    fig, ax = plt.subplots(
        2 * len(workload_distributions),
        len(tp_ep_list),
        figsize=(5 * len(tp_ep_list), 5 * 2 * len(workload_distributions)),
        squeeze=False,
    )
    for workload_distribution_id, workload_distribution in enumerate(workload_distributions):
        for i, (tp, ep) in enumerate(tp_ep_list):
            for quant_mode in database._moe_data.keys():
                if quant_mode == common.MoEQuantMode.w4a16_mxfp4:
                    preferred_profile = MoeChartProfile(
                        topk=4, num_experts=128, hidden_size=2880, inter_size=2880
                    )
                else:
                    preferred_profile = MoeChartProfile(
                        topk=8, num_experts=256, hidden_size=7168, inter_size=2048
                    )

                weight_bits = int(quant_mode.value.memory * 8)

                def profile_is_queryable(profile):
                    if (profile.inter_size // tp) % (256 // weight_bits) != 0:
                        return False
                    return not (
                        quant_mode == common.MoEQuantMode.fp8_block and (profile.inter_size // tp) % 128 != 0
                    )

                profiles = select_moe_chart_profiles(
                    database._moe_data[quant_mode],
                    workload_distribution=workload_distribution,
                    moe_tp_size=tp,
                    moe_ep_size=ep,
                    target_tokens=m_list,
                    preferred=preferred_profile,
                    predicate=profile_is_queryable,
                )
                for profile in profiles:
                    sol_math_list = []
                    sol_mem_list = []
                    for m in m_list:
                        sol_time, sol_math, sol_mem = reference.query_moe(
                            num_tokens=m,
                            hidden_size=profile.hidden_size,
                            inter_size=profile.inter_size,
                            topk=profile.topk,
                            num_experts=profile.num_experts,
                            moe_tp_size=tp,
                            moe_ep_size=ep,
                            quant_mode=quant_mode,
                            workload_distribution=workload_distribution,
                            database_mode=DatabaseMode.SOL_FULL,
                        )
                        # TODO: fix query_moe for all combos and remove try/except
                        try:
                            db_time = reference.query_moe(
                                num_tokens=m,
                                hidden_size=profile.hidden_size,
                                inter_size=profile.inter_size,
                                topk=profile.topk,
                                num_experts=profile.num_experts,
                                moe_tp_size=tp,
                                moe_ep_size=ep,
                                quant_mode=quant_mode,
                                workload_distribution=workload_distribution,
                                database_mode=DatabaseMode.SILICON,
                            )
                        except Exception as e:
                            print(f"Error querying moe: {e}")
                            break
                        percentage_of_math = sol_math / db_time
                        percentage_of_mem = sol_mem / db_time
                        sol_math_list.append(percentage_of_math)
                        sol_mem_list.append(percentage_of_mem)

                    if len(m_list) == len(sol_math_list) and len(m_list) == len(sol_mem_list):
                        profile_suffix = "" if profile == preferred_profile else f" [{profile.label}]"
                        color_key = (quant_mode, profile)
                        if color_key not in color_by_series:
                            color_by_series[color_key] = color_list[len(color_by_series) % len(color_list)]
                        color = color_by_series[color_key]
                        ax[workload_distribution_id * 2, i].plot(
                            m_list, sol_math_list, color=color, label=f"{quant_mode}{profile_suffix} math"
                        )
                        ax[workload_distribution_id * 2 + 1, i].plot(
                            m_list,
                            sol_mem_list,
                            color=color,
                            linestyle="--",
                            label=f"{quant_mode}{profile_suffix} mem",
                        )
            if workload_distribution != "balanced":
                workload_distribution_title = workload_distribution + "  vs balanced sol"
            else:
                workload_distribution_title = workload_distribution

            ax[workload_distribution_id * 2, i].set_title(f"{workload_distribution_title} \ntp={tp} ep={ep}")
            ax[workload_distribution_id * 2, i].set_xlabel("num_tokens")
            ax[workload_distribution_id * 2, i].set_ylabel("math sol %")
            # ax[0,i].set_ylim(0,1)
            ax[workload_distribution_id * 2, i].legend()
            ax[workload_distribution_id * 2 + 1, i].set_xlabel("num_tokens")
            ax[workload_distribution_id * 2 + 1, i].set_ylabel("mem sol %")
            # ax[1,i].set_ylim(0,1)
            ax[workload_distribution_id * 2 + 1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_moe(database, reference)

In [ ]:
def visualize_allreduce(database, reference):
    database._custom_allreduce_data.raise_if_not_loaded()

    quant_mode = common.CommQuantMode.half
    m_list = [
        2**0,
        2**1,
        2**2,
        2**4,
        2**6,
        2**8,
        2**10,
        2**12,
        2**14,
        2**16,
        2**18,
        2**20,
        2**22,
        2**24,
        2**26,
        2**28,
        2**30,
        2**32,
    ]
    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    if database.system in ["gb200", "gb300"]:
        tp_list = [2, 4]
    else:
        tp_list = [2, 4, 8, 16, 32]
    fig, ax = plt.subplots(1, len(tp_list), figsize=(5 * len(tp_list), 5 * 1))
    for i, tp_size in enumerate(tp_list):
        sol_list = []
        for m in m_list:
            sol_time, sol_math, sol_mem = reference.query_custom_allreduce(
                quant_mode, tp_size, m, database_mode=DatabaseMode.SOL_FULL
            )
            db_time = reference.query_custom_allreduce(quant_mode, tp_size, m, database_mode=DatabaseMode.SILICON)
            percentage_of_sol = sol_time / db_time
            sol_list.append(percentage_of_sol)
        ax[i].plot(m_list, sol_list, color=color_list[i], label=f"{tp_size}")
        ax[i].set_title(f"tp {tp_size}")
        # ax[i].set_xscale('log', base=2)
        ax[i].set_xlabel("message_size")
        ax[i].set_ylabel("sol %")
        # ax[i].set_ylim(0,1)
        ax[i].legend()
    plt.tight_layout()
    plt.show()


try:
    visualize_allreduce(database, reference)
except (PerfDataNotAvailableError, ValueError) as e:
    print(e)

In [ ]:
def visualize_nccl(database, reference, operation="all_gather"):
    quant_mode = common.CommQuantMode.half
    m_list = [
        2**0,
        2**1,
        2**2,
        2**4,
        2**6,
        2**8,
        2**10,
        2**12,
        2**14,
        2**16,
        2**18,
        2**20,
        2**22,
        2**24,
        2**26,
        2**28,
        2**30,
        2**32,
    ]
    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    num_gpu_list = [2, 4, 8, 16, 32, 64]
    fig, ax = plt.subplots(1, len(num_gpu_list), figsize=(5 * len(num_gpu_list), 5 * 1))
    for i, num_gpu in enumerate(num_gpu_list):
        sol_list = []
        for m in m_list:
            sol_time, sol_math, sol_mem = reference.query_nccl(
                quant_mode, num_gpu, operation, m, database_mode=DatabaseMode.SOL_FULL
            )
            db_time = reference.query_nccl(quant_mode, num_gpu, operation, m, database_mode=DatabaseMode.SILICON)
            percentage_of_sol = sol_time / db_time
            sol_list.append(percentage_of_sol)
        ax[i].plot(m_list, sol_list, color=color_list[i], label=f"{num_gpu}")
        ax[i].set_title(f"{operation} num_gpu {num_gpu}")
        # ax[i].set_xscale('log', base=2)
        ax[i].set_xlabel("message_size")
        ax[i].set_ylabel("sol %")
        # ax[i].set_ylim(0,1)
        ax[i].legend()
    plt.tight_layout()
    plt.show()


def _op_data_loaded(op_data):
    return op_data is not None and getattr(op_data, "loaded", False)


op_list = ["all_gather", "all_reduce", "alltoall", "reduce_scatter"]
if _op_data_loaded(getattr(database, "_nccl_data", None)) or _op_data_loaded(getattr(database, "_oneccl_data", None)):
    for op in op_list:
        visualize_nccl(database, reference, operation=op)
else:
    print("No NCCL/oneCCL data available - skipping")

In [ ]:
# NOTE(#1357): re-oracled in PR-5. This chart walks the raw per-phase alltoall
# table (prepare/dispatch/combine/combine_lp) directly: every plotted point is
# a collected row, so the SILICON value IS the row's latency, and the SOL
# reference is the closed-form wire model below (the same formula the engine
# uses; python mirror verified 0-mismatch against the retired facade). No
# per-call query surface involved.
def _alltoall_leaf_latency(leaf):
    return leaf["latency"] if isinstance(leaf, dict) else float(leaf)


def _alltoall_sol_ms(op_name, num_tokens, hidden_size, topk, num_experts, moe_ep_size, quant_mode, node_num, spec):
    """Closed-form SOL for one alltoall phase (ms).

    prepare moves topk*4B of routing metadata per token; dispatch sends each
    token once per unique remote rank (min(topk, num_experts, ep-1)) at the
    quant precision; combine returns results in bf16 (2B) or fp4 (0.5B for the
    low-precision variant). Wire bandwidth is the intra/inter-node link from
    the system spec.
    """
    bw = spec["node"]["inter_node_bw"] if node_num > 1 else spec["node"]["intra_node_bw"]
    remote_ranks = min(topk, num_experts, moe_ep_size - 1)
    if op_name == "alltoall_prepare":
        data_bytes = num_tokens * topk * 4
    elif "combine" in op_name:
        bytes_per_element = 0.5 if "low_precision" in op_name else 2
        data_bytes = num_tokens * remote_ranks * hidden_size * bytes_per_element
    else:
        data_bytes = num_tokens * remote_ranks * hidden_size * quant_mode.value.memory
    return data_bytes / bw * 1000


def visualize_trtllm_alltoall(database):
    """Visualize TRT-LLM AlltoAll communication latency and sol%.

    Layout: rows = op phases (prepare/dispatch/combine/combine_lp) with latency and sol%,
            cols = ep_sizes, lines = moe_dtype (quant mode).

    """
    alltoall_data = database._trtllm_alltoall_data
    if not alltoall_data.loaded:
        print("No trtllm alltoall data available (data not loaded)")
        return
    if not alltoall_data:
        print("No trtllm alltoall data available (empty)")
        return

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]

    for kernel_source in alltoall_data:
        kernel_data = alltoall_data[kernel_source]

        op_names_set = set()
        ep_sizes_set = set()
        quant_modes_set = set()

        for op_name in kernel_data:
            op_names_set.add(op_name)
            for qm in kernel_data[op_name]:
                quant_modes_set.add(qm)
                for nn in kernel_data[op_name][qm]:
                    for hs in kernel_data[op_name][qm][nn]:
                        for tk in kernel_data[op_name][qm][nn][hs]:
                            for ne in kernel_data[op_name][qm][nn][hs][tk]:
                                for ep in kernel_data[op_name][qm][nn][hs][tk][ne]:
                                    ep_sizes_set.add(ep)

        op_order = [
            "alltoall_prepare",
            "alltoall_dispatch",
            "alltoall_combine",
            "alltoall_combine_low_precision",
        ]
        op_names = [op for op in op_order if op in op_names_set]
        ep_sizes = sorted(ep_sizes_set)
        quant_modes = sorted(quant_modes_set, key=str)

        if not op_names or not ep_sizes:
            continue

        sol_ops = {"alltoall_dispatch", "alltoall_combine", "alltoall_combine_low_precision"}
        rows = []
        for op in op_names:
            rows.append((op, "latency"))
            if op in sol_ops:
                rows.append((op, "sol %"))

        n_rows = len(rows)
        n_cols = max(len(ep_sizes), 1)
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(5 * n_cols, 4 * n_rows),
            squeeze=False,
        )
        fig.suptitle(
            f"{database.system.upper()} - {database.backend.upper()} {database.version}"
            f" - TRT-LLM AlltoAll Sanity Chart ({kernel_source})",
            fontsize=14,
            fontweight="bold",
        )

        for ri, (op_name, metric) in enumerate(rows):
            for ci, target_ep in enumerate(ep_sizes):
                ax = axes[ri][ci]
                cid = 0
                for qm in quant_modes:
                    # Collect the collected-token leaves keyed by shape tuple so
                    # each plotted line uses its own exact rows.
                    shape_map = {}  # (nn, h, t, ne) -> {num_tokens: leaf}
                    if qm in kernel_data.get(op_name, {}):
                        for nn in kernel_data[op_name][qm]:
                            for h in kernel_data[op_name][qm][nn]:
                                for t in kernel_data[op_name][qm][nn][h]:
                                    for ne in kernel_data[op_name][qm][nn][h][t]:
                                        if target_ep not in kernel_data[op_name][qm][nn][h][t][ne]:
                                            continue
                                        key = (nn, h, t, ne)
                                        shape_map.setdefault(key, {}).update(
                                            kernel_data[op_name][qm][nn][h][t][ne][target_ep]
                                        )

                    if not shape_map:
                        cid += 1
                        continue

                    for (nn_val, hs, tk, ne_val), leaves in shape_map.items():
                        tokens = sorted(leaves)
                        base_label = qm.name if hasattr(qm, "name") else str(qm)
                        label = (
                            f"{base_label} nn={nn_val} hs={hs} tk={tk} ne={ne_val}"
                            if len(shape_map) > 1
                            else base_label
                        )

                        if metric == "latency":
                            vals = [_alltoall_leaf_latency(leaves[nt]) for nt in tokens]
                        else:
                            vals = []
                            for nt in tokens:
                                sol_time = _alltoall_sol_ms(
                                    op_name, nt, hs, tk, ne_val, target_ep, qm, nn_val, database.system_spec
                                )
                                db_time = _alltoall_leaf_latency(leaves[nt])
                                vals.append(sol_time / db_time if db_time > 0 else 0)

                        ax.plot(
                            tokens,
                            vals,
                            color=color_list[cid % len(color_list)],
                            label=label,
                            marker=".",
                            markersize=3,
                        )
                        cid += 1

                ax.set_title(f"{op_name}\nep_size={target_ep}")
                ax.set_xlabel("num_tokens")
                ax.set_ylabel(metric)
                ax.legend(fontsize="small")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()


if "AIC_VALIDATE_SYSTEM" in os.environ and (system, backend) != ("gb200", "trtllm"):
    # Parameterized CI run for another combo: skip the hardcoded GB200 load so
    # every per-system validation run does not pay to warm an unrelated database.
    print("Skipping trtllm alltoall visualization (parameterized run for a different system)")
else:
    alltoall_database = get_database(system="gb200", backend="trtllm", version="1.3.0rc10", allow_unlisted_version=True)  # sole trtllm_alltoall coordinate (kept donor data)
    if alltoall_database is None:
        print("GB200 trtllm 1.3.0rc10 database unavailable – skipping alltoall visualization")
    else:
        warm_all_op_data(alltoall_database)
        visualize_trtllm_alltoall(alltoall_database)
